In [1]:
import pandas as pd

df = pd.read_csv("maize_yield_rwanda_rw.csv")

print("Dataset loaded successfully!")
print(df.shape)
display(df.head())

Dataset loaded successfully!
(1426, 20)


,record_id,year,season,province,district,altitude_m,rainfall_mm,avg_temp_c,soil_ph,soil_type,seed_variety,farming_practice,npk_kg_ha,urea_kg_ha,manure_t_ha,plant_density_per_ha,farm_area_ha,irrigation,pest_incidence_index,yield_t_ha
0,RW-MZ-00001,2015,A,Eastern,Bugesera,1349,NaN,22.9,NaN,Clay,RHM104,Improved,117.8,57.4,1.41,61803,0.63,No,NaN,2.71
1,RW-MZ-00002,2015,A,Eastern,Bugesera,1427,489.8,22.1,6.02,Volcanic,RHM104,Improved,38.1,62.7,3.69,52919,0.17,No,0.339,4.09
2,RW-MZ-00003,2015,A,Northern,Burera,2033,722.6,16.3,5.73,Clay loam,Local,Improved,121.1,50.7,5.18,54406,0.22,No,0.049,2.58
3,RW-MZ-00004,2015,A,Northern,Burera,2026,794.4,16.4,5.20,Clay loam,ZM621,Improved,93.7,60.4,1.68,48119,0.48,No,0.171,2.45
4,RW-MZ-00005,2015,A,Northern,Gakenke,1974,644.2,18.3,5.92,Loam,Local,Improved,84.1,70.4,0.36,56787,1.16,No,0.001,2.09


In [3]:
print("Missing values before cleaning:")

display(df.isnull().sum())

Missing values before cleaning:


record_id                0
year                     0
season                   0
province                 0
district                 0
altitude_m               0
rainfall_mm             11
avg_temp_c               0
soil_ph                 14
soil_type                0
seed_variety             0
farming_practice         0
npk_kg_ha                0
urea_kg_ha               0
manure_t_ha              0
plant_density_per_ha     0
farm_area_ha             0
irrigation               0
pest_incidence_index    14
yield_t_ha               0
dtype: int64

In [4]:
clean_df = df.copy()

print("Working copy created.")
print("Original dataset remains unchanged.")

Working copy created.
Original dataset remains unchanged.


In [6]:
import numpy as np
import pandas as pd

In [8]:
# Remove unnecessary spaces from column names
clean_df.columns = clean_df.columns.str.strip()

# Remove unnecessary spaces from categorical values
categorical_columns = clean_df.select_dtypes(
    include=["object", "string"]
).columns

for column in categorical_columns:
    clean_df[column] = clean_df[column].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

# Convert empty strings into missing values
clean_df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

print("Text formatting completed.")

Text formatting completed.


In [9]:

numeric_columns = [
    "year",
    "altitude_m",
    "rainfall_mm",
    "avg_temp_c",
    "soil_ph",
    "npk_kg_ha",
    "urea_kg_ha",
    "manure_t_ha",
    "plant_density_per_ha",
    "farm_area_ha",
    "pest_incidence_index",
    "yield_t_ha"
]

conversion_log = []

for column in numeric_columns:
    if column in clean_df.columns:
        before_missing = clean_df[column].isna().sum()

        clean_df[column] = pd.to_numeric(
            clean_df[column],
            errors="coerce"
        )

        after_missing = clean_df[column].isna().sum()

        conversion_log.append({
            "Column": column,
            "New missing values from conversion":
                after_missing - before_missing
        })

display(pd.DataFrame(conversion_log))

,Column,New missing values from conversion
0,year,0
1,altitude_m,0
2,rainfall_mm,0
3,avg_temp_c,0
4,soil_ph,0
5,npk_kg_ha,0
6,urea_kg_ha,0
7,manure_t_ha,0
8,plant_density_per_ha,0
9,farm_area_ha,0


In [10]:
display(clean_df[numeric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
year,1426.0,2019.506311,2.855643,2015.000,2017.000,2020.000,2022.00000,2024.000
altitude_m,1426.0,1704.321879,221.267632,1243.000,1507.000,1702.500,1879.75000,2341.000
rainfall_mm,1415.0,516.165371,155.676106,150.000,419.350,530.600,627.45000,933.300
avg_temp_c,1426.0,19.404137,1.645486,15.400,18.025,19.400,20.70000,23.000
soil_ph,1412.0,5.835156,0.443019,4.500,5.530,5.840,6.13000,7.200
npk_kg_ha,1426.0,66.991304,47.311529,0.000,26.300,62.450,104.97500,208.100
urea_kg_ha,1426.0,38.767602,31.571860,0.000,11.500,33.100,61.87500,135.200
manure_t_ha,1426.0,2.083415,1.361537,0.000,1.060,2.005,3.02750,6.860
plant_density_per_ha,1426.0,46322.309958,9695.817375,19337.000,38471.500,46901.000,53806.25000,72248.000
farm_area_ha,1426.0,0.802104,0.511796,0.100,0.460,0.670,0.98000,4.220


In [11]:
range_checks = {
    "rainfall_mm": clean_df["rainfall_mm"] < 0,
    "avg_temp_c": (
        (clean_df["avg_temp_c"] < -10) |
        (clean_df["avg_temp_c"] > 60)
    ),
    "soil_ph": (
        (clean_df["soil_ph"] < 0) |
        (clean_df["soil_ph"] > 14)
    ),
    "farm_area_ha": clean_df["farm_area_ha"] <= 0,
    "yield_t_ha": clean_df["yield_t_ha"] < 0
}

for column, condition in range_checks.items():
    print(column, "potentially invalid values:", condition.sum())

rainfall_mm potentially invalid values: 0
avg_temp_c potentially invalid values: 0
soil_ph potentially invalid values: 0
farm_area_ha potentially invalid values: 0
yield_t_ha potentially invalid values: 0


In [13]:
import os

# Create docs folder if it does not exist
os.makedirs("../docs", exist_ok=True)

# Save cleaning log
cleaning_log.to_csv(
    "../docs/cleaning_log.csv",
    index=False
)

print("Cleaning log saved successfully!")

Cleaning log saved successfully!


In [14]:
target = "yield_t_ha"

# Remove records where the target is missing
before_rows = len(clean_df)

clean_df = clean_df.dropna(subset=[target]).copy()

removed_rows = before_rows - len(clean_df)

print("Rows removed due to missing target:", removed_rows)

# Separate target and input features
X = clean_df.drop(
    columns=["yield_t_ha", "record_id"],
    errors="ignore"
)

y = clean_df["yield_t_ha"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Rows removed due to missing target: 0
Features shape: (1426, 18)
Target shape: (1426,)


In [17]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

In [18]:
print("Training data:", X_train.shape)
print("Temporary data:", X_temp.shape)
print("Training labels:", y_train.shape)
print("Temporary labels:", y_temp.shape)

Training data: (998, 18)
Temporary data: (428, 18)
Training labels: (998,)
Temporary labels: (428,)


In [19]:
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool", "string"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


Numerical features:
['year', 'altitude_m', 'rainfall_mm', 'avg_temp_c', 'soil_ph', 'npk_kg_ha', 'urea_kg_ha', 'manure_t_ha', 'plant_density_per_ha', 'farm_area_ha', 'pest_incidence_index']

Categorical features:
['season', 'province', 'district', 'soil_type', 'seed_variety', 'farming_practice', 'irrigation']


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipelines created successfully!")

Preprocessing pipelines created successfully!


In [27]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

In [28]:
# Training data
X_train_processed = preprocessor.fit_transform(X_train)

# Validation data
X_val_processed = preprocessor.transform(X_val)

# Testing data
X_test_processed = preprocessor.transform(X_test)

print("All datasets processed successfully!")

All datasets processed successfully!


In [29]:
feature_names = preprocessor.get_feature_names_out()

In [30]:
feature_names = preprocessor.get_feature_names_out()

X_train_final = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_val_final = pd.DataFrame(
    X_val_processed,
    columns=feature_names,
    index=X_val.index
)

X_test_final = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_train_final.head())

,numeric__year,numeric__altitude_m,numeric__rainfall_mm,numeric__avg_temp_c,numeric__soil_ph,numeric__npk_kg_ha,numeric__urea_kg_ha,numeric__manure_t_ha,numeric__plant_density_per_ha,numeric__farm_area_ha,...,categorical__seed_variety_Local,categorical__seed_variety_PAN691,categorical__seed_variety_RHM104,categorical__seed_variety_WH505,categorical__seed_variety_ZM607,categorical__seed_variety_ZM621,categorical__farming_practice_Improved,categorical__farming_practice_Traditional,categorical__irrigation_No,categorical__irrigation_Yes
390,-0.860897,0.772074,0.014741,-0.430069,-1.733868,-1.391477,-0.216704,-0.209508,-1.987868,-1.338987,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
573,-0.165490,0.961328,1.530772,-0.976832,0.254993,-0.854603,0.587311,-0.953178,0.576370,5.997153,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
807,0.182213,-1.647674,-1.247534,2.060740,0.735063,-0.704877,-0.261910,-0.187419,-1.918933,-0.539036,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1228,1.225323,0.141227,0.166995,-0.551572,-0.247938,-1.391477,-1.204771,0.924404,-0.806228,-0.519525,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
585,-0.165490,0.055612,0.191720,0.116694,1.146551,-0.587235,-0.736569,0.210187,-1.619497,0.768202,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [31]:
print("Training features:", X_train_final.shape)
print("Validation features:", X_val_final.shape)
print("Testing features:", X_test_final.shape)

print("\nTraining target:", y_train.shape)
print("Validation target:", y_val.shape)
print("Testing target:", y_test.shape)

Training features: (998, 64)
Validation features: (214, 64)
Testing features: (214, 64)

Training target: (998,)
Validation target: (214,)
Testing target: (214,)


In [32]:
print("Missing values in training:",
      X_train_final.isnull().sum().sum())

print("Missing values in validation:",
      X_val_final.isnull().sum().sum())

print("Missing values in testing:",
      X_test_final.isnull().sum().sum())

Missing values in training: 0
Missing values in validation: 0
Missing values in testing: 0


In [34]:
import os

# Create the processed data folder
os.makedirs("../data/processed", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [37]:
# Save training data
X_train_final.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

# Save validation data
X_val_final.to_csv(
    "../data/processed/X_val.csv",
    index=False
)

# Save testing data
X_test_final.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [36]:
# Save training data
X_train_final.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

# Save validation data
X_val_final.to_csv(
    "../data/processed/X_val.csv",
    index=False
)

# Save testing data
X_test_final.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [38]:
summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Testing"],
    "Number of Records": [
        len(X_train_final),
        len(X_val_final),
        len(X_test_final)
    ],
    "Number of Features": [
        X_train_final.shape[1],
        X_val_final.shape[1],
        X_test_final.shape[1]
    ]
})

display(summary)

,Dataset,Number of Records,Number of Features
0,Training,998,64
1,Validation,214,64
2,Testing,214,64


In [39]:
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
display(df.isnull().sum())

print("\nUnique values:")
display(df.nunique())


Duplicate rows: 0

Missing values:


record_id                0
year                     0
season                   0
province                 0
district                 0
altitude_m               0
rainfall_mm             11
avg_temp_c               0
soil_ph                 14
soil_type                0
seed_variety             0
farming_practice         0
npk_kg_ha                0
urea_kg_ha               0
manure_t_ha              0
plant_density_per_ha     0
farm_area_ha             0
irrigation               0
pest_incidence_index    14
yield_t_ha               0
dtype: int64


Unique values:


record_id               1426
year                      10
season                     3
province                   5
district                  30
altitude_m               680
rainfall_mm             1225
avg_temp_c                76
soil_ph                  211
soil_type                  5
seed_variety               6
farming_practice           2
npk_kg_ha                925
urea_kg_ha               732
manure_t_ha              463
plant_density_per_ha    1391
farm_area_ha             216
irrigation                 2
pest_incidence_index     563
yield_t_ha               308
dtype: int64

In [40]:
target = "yield_t_ha"

# Remove duplicate records
df = df.drop_duplicates().copy()

# Separate features and target
X = df.drop(columns=[target, "record_id"], errors="ignore")
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1426, 18)
Target shape: (1426,)
